In [ ]:
!pip install langchain -qU
!pip install langchain-openai -qU
!pip install langchain-chroma -qU
!pip install langchain_community -qU



In [ ]:
import os
from google.colab import files
from google.colab import userdata

# 1. Upload the PDF
print("Please upload your file:")
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]



try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    os.environ['GOOGLE_API_KEY'] = userdata.get('EMVEDDING_KEY')
    print(" API Keys loaded")
except:

    print("error")

Please upload your file:


Saving medilink_chatbot.pdf to medilink_chatbot (2).pdf
 API Keys loaded


In [ ]:
! pip install langchain-classic

In [ ]:
!pip install -U langchain-google-genai

In [ ]:
!pip install pypdf -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 18.8 MB/s eta 0:00:00


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# 1. Load the PDF
print(f"Loading {pdf_filename}...")
loader = PyPDFLoader(pdf_filename)
docs = loader.load()

# 2. Split Text (Chunks)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(docs)
print(f" Split document into {len(splits)} chunks.")

# 3. Initialize Embedding Model

embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# 4. Create & SAVE Vector Store

print("Creating Vector Database")
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embedding_model,
    persist_directory="./medilink_db"
)

print("Indexing Complete!")

Loading medilink_chatbot (2).pdf...
 Split document into 8 chunks.
Creating Vector Database
Indexing Complete!


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. Setup LLM
llm = ChatOpenAI(
    model="tngtech/deepseek-r1t2-chimera:free",
    openai_api_base="https://openrouter.ai/api/v1",
)

# 2. Setup Retriever
retriever = vectorstore.as_retriever()



contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# This specific chain reformulates the query
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)




qa_system_prompt = (
    "You are the MediLink assistant. Use the context below to answer. "
    "IMPORTANT: Answer directly and do not mention 'context' or 'documents'.\n\n"
    "{context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)


rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)



# Simple in-memory storage for chat history
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# The final "Runnable" that handles everything
conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)



session_id = "user_kavinda_test_1"



response1 = conversational_rag_chain.invoke(
    {"input": "What is MediLink?"},
    config={"configurable": {"session_id": session_id}}
)
print(f"User: What is MediLink?")
print(f"AI: {response1['answer']}\n")



response2 = conversational_rag_chain.invoke(
    {"input": "Does it have a triage feature?"},
    config={"configurable": {"session_id": session_id}}
)
print(f"User: Does it have a triage feature?")
print(f"AI: {response2['answer']}")

User: What is MediLink?
AI: MediLink is an AI-integrated telemedicine platform in Sri Lanka that uses intelligent symptom analysis to connect patients with the right medical specialist before they travel or book appointments. It solves the problem of patients making unnecessary trips to hospitals by providing preliminary diagnoses and specialist recommendations through its AI triage system. The platform primarily serves rural communities and helps doctors receive pre-screened cases efficiently.

User: Does it have a triage feature?
AI: **Yes.** MediLink’s core feature is its **AI-powered triage system**:  
1. You input your symptoms (via forms or free text).  
2. The AI analyzes them against medical data to generate a "Preliminary Diagnosis."  
3. It instantly recommends the **correct specialist** (e.g., Dermatologist for a skin rash) to ensure you skip guesswork and reach the right doctor **from the first visit.**  

This eliminates mismatched appointments and streamlines care — espec